# Name Generation using LSTMs
In this project, we'll build a character-level language model using an LSTM network to generate new, realistic-sounding names based on the `NationalNames.csv` dataset you have.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
import random
import os
print('TensorFlow version:', tf.__version__)

## 1. Load and Prepare the Dataset

In [ ]:
# Load the dataset
# Ensure 'NationalNames.csv' is in the same directory
df = pd.read_csv('NationalNames.csv')
print(df.head())

# Extract unique names and convert them to lowercase
names = df['Name'].dropna().unique()
names = [name.lower() for name in names]
print(f"\nTotal unique names: {len(names)}")
print("Sample names:", names[:5])

## 2. Character-Level Tokenization
We need to convert characters into integers so the neural network can process them.

In [ ]:
# Create vocabulary of unique characters
chars = sorted(list(set(''.join(names))))
chars.insert(0, '\n') # Add newline as a stop character
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")

# Create mapping from characters to integers and vice versa
char2idx = {char: idx for idx, char in enumerate(chars)}
idx2char = {idx: char for idx, char in enumerate(chars)}

## 3. Create Training Sequences
We will train the model to predict the next character given a sequence of preceding characters.

In [ ]:
# Set a maximum sequence length (e.g., longest name length)
max_len = max([len(name) for name in names])

X = []
y = []

# We'll use a subset of names (first 10,000) for faster training in this example
sample_names = names[:10000]

for name in sample_names:
    name = name + '\n' # Append stop character
    for i in range(1, len(name)):
        seq_in = name[:i]
        seq_out = name[i]
        X.append([char2idx[char] for char in seq_in])
        y.append(char2idx[seq_out])

# Pad sequences so they are all the same length
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')
y = np.array(y)

print("Shape of X_padded:", X_padded.shape)
print("Shape of y:", y.shape)

## 4. Build the LSTM Model

In [ ]:
model = Sequential([
    Embedding(vocab_size, 50, input_length=max_len),
    LSTM(128, return_sequences=False),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 5. Train the Model
(Note: This might take a few minutes depending on your hardware)

In [ ]:
# Train for a few epochs
history = model.fit(X_padded, y, epochs=10, batch_size=64, validation_split=0.1)

## 6. Generate New Names

In [ ]:
def generate_name(seed_letter, num_chars=15):
    current_seq = [char2idx[seed_letter.lower()]]
    generated_name = seed_letter.upper()
    
    for _ in range(num_chars):
        padded_seq = pad_sequences([current_seq], maxlen=max_len, padding='pre')
        
        # Predict probabilities for the next character
        preds = model.predict(padded_seq, verbose=0)[0]
        
        # Sample from the probabilities (adds randomness instead of argmax)
        next_idx = np.random.choice(vocab_size, p=preds)
        next_char = idx2char[next_idx]
        
        if next_char == '\n':
            break
            
        generated_name += next_char
        current_seq.append(next_idx)
        
    return generated_name

# Let's generate a few names starting with random letters!
print("Generating names...")
for letter in ['A', 'M', 'E', 'L', 'J']:
    print(f"{letter} -> {generate_name(letter)}")